# Exploratory Data Analysis (EDA)
## Telco Customer Churn Dataset

This notebook performs comprehensive exploratory data analysis on the Telco Customer Churn dataset.

### Contents:
1. Data Loading & Overview
2. Data Quality Check
3. Univariate Analysis
4. Bivariate Analysis
5. Correlation Analysis
6. Key Insights & Conclusions

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('Libraries imported successfully!')

## 1. Data Loading & Overview

In [ ]:
# Load the dataset
data_path = Path('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(data_path)

print(f'Dataset Shape: {df.shape}')
print(f'Number of Customers: {df.shape[0]:,}')
print(f'Number of Features: {df.shape[1]}')

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Data types and info
df.info()

In [ ]:
# Statistical summary for numerical columns
df.describe()

In [ ]:
# Statistical summary for categorical columns
df.describe(include='object')

## 2. Data Quality Check

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print('Missing Values Analysis:')
print(missing_df[missing_df['Missing Count'] > 0])

if missing_df['Missing Count'].sum() == 0:
    print('\nNo missing values in the dataset!')

In [ ]:
# Check TotalCharges column (often has whitespace issues)
print('TotalCharges data type:', df['TotalCharges'].dtype)

# Convert to numeric and check for issues
df['TotalCharges_numeric'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'\nNon-numeric TotalCharges values: {df["TotalCharges_numeric"].isnull().sum()}')

# View problematic rows
print('\nRows with invalid TotalCharges:')
df[df['TotalCharges_numeric'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

# Check duplicate customer IDs
duplicate_ids = df['customerID'].duplicated().sum()
print(f'Duplicate customer IDs: {duplicate_ids}')

In [ ]:
# Check unique values for each column
print('Unique values per column:\n')
for col in df.columns:
    unique_count = df[col].nunique()
    print(f'{col}: {unique_count}')

## 3. Univariate Analysis

### 3.1 Target Variable Analysis

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
churn_counts = df['Churn'].value_counts()
axes[0].bar(churn_counts.index, churn_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Distribution (Count)', fontsize=14)
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')

# Add count labels
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'], startangle=90, explode=[0, 0.05])
axes[1].set_title('Churn Distribution (%)', fontsize=14)

plt.tight_layout()
plt.show()

print(f"\nChurn Rate: {df['Churn'].value_counts(normalize=True)['Yes']:.2%}")

### 3.2 Numerical Features Distribution

In [ ]:
# Numerical columns
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges_numeric']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i].set_title(f'{col} Distribution', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.2f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for numerical features
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(numerical_cols):
    sns.boxplot(data=df, y=col, ax=axes[i], color='lightblue')
    axes[i].set_title(f'{col} Box Plot')

plt.tight_layout()
plt.show()

### 3.3 Categorical Features Distribution

In [ ]:
# Define categorical columns
categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
                   'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                   'Contract', 'PaperlessBilling', 'PaymentMethod']

# Plot categorical distributions
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    value_counts = df[col].value_counts()
    axes[i].barh(value_counts.index, value_counts.values, color='steelblue')
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 4. Bivariate Analysis (Features vs Churn)

### 4.1 Numerical Features vs Churn

In [ ]:
# Numerical features by Churn
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(numerical_cols):
    sns.boxplot(data=df, x='Churn', y=col, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'{col} by Churn')

plt.tight_layout()
plt.show()

In [ ]:
# KDE plots for numerical features by Churn
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(numerical_cols):
    for churn_status in ['No', 'Yes']:
        subset = df[df['Churn'] == churn_status][col].dropna()
        color = '#2ecc71' if churn_status == 'No' else '#e74c3c'
        sns.kdeplot(subset, ax=axes[i], label=f'Churn: {churn_status}', color=color, fill=True, alpha=0.3)
    axes[i].set_title(f'{col} Distribution by Churn')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Statistical comparison
print('Mean values by Churn status:\n')
print(df.groupby('Churn')[numerical_cols].mean().round(2))

### 4.2 Categorical Features vs Churn

In [ ]:
# Churn rate by categorical features
fig, axes = plt.subplots(4, 4, figsize=(20, 18))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    churn_rate = churn_rate.sort_values(ascending=True)
    
    colors = ['#e74c3c' if rate > 30 else '#f39c12' if rate > 20 else '#2ecc71' 
              for rate in churn_rate.values]
    
    axes[i].barh(churn_rate.index, churn_rate.values, color=colors)
    axes[i].set_title(f'Churn Rate by {col}', fontsize=11)
    axes[i].set_xlabel('Churn Rate (%)')
    axes[i].axvline(x=df['Churn'].apply(lambda x: x == 'Yes').mean() * 100, 
                    color='navy', linestyle='--', label='Overall')

plt.tight_layout()
plt.show()

In [ ]:
# Key categorical features deep dive
key_features = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[i].set_title(f'Churn Rate by {col}', fontsize=12)
    axes[i].set_ylabel('Percentage %')
    axes[i].set_xlabel(col)
    axes[i].legend(title='Churn', loc='upper right')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4.3 Tenure Analysis

In [ ]:
# Create tenure groups
bins = [0, 12, 24, 48, 72]
labels = ['0-12 months', '12-24 months', '24-48 months', '48+ months']
df['TenureGroup'] = pd.cut(df['tenure'], bins=bins, labels=labels, include_lowest=True)

# Churn rate by tenure group
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count by tenure group
tenure_churn = pd.crosstab(df['TenureGroup'], df['Churn'])
tenure_churn.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Customer Count by Tenure Group')
axes[0].set_xlabel('Tenure Group')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Churn rate by tenure group
churn_rate_tenure = df.groupby('TenureGroup')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
axes[1].bar(churn_rate_tenure.index.astype(str), churn_rate_tenure.values, 
           color=['#e74c3c', '#f39c12', '#f1c40f', '#2ecc71'], edgecolor='black')
axes[1].set_title('Churn Rate by Tenure Group')
axes[1].set_xlabel('Tenure Group')
axes[1].set_ylabel('Churn Rate (%)')

# Add value labels
for i, v in enumerate(churn_rate_tenure.values):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Create binary/numeric versions for correlation
df_corr = df.copy()

# Convert Churn to binary
df_corr['Churn_Binary'] = df_corr['Churn'].map({'No': 0, 'Yes': 1})

# Use TotalCharges numeric
df_corr['TotalCharges'] = df_corr['TotalCharges_numeric']

# Encode categorical variables for correlation
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    if col in df_corr.columns:
        df_corr[f'{col}_enc'] = df_corr[col].map({'No': 0, 'Yes': 1, 'Female': 0, 'Male': 1})

# Numerical columns for correlation
corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
             'Partner_enc', 'Dependents_enc', 'PhoneService_enc', 'PaperlessBilling_enc', 'Churn_Binary']

# Calculate correlation matrix
corr_matrix = df_corr[corr_cols].corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation with Churn
churn_corr = corr_matrix['Churn_Binary'].drop('Churn_Binary').sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if x > 0 else '#3498db' for x in churn_corr.values]
plt.barh(churn_corr.index, churn_corr.values, color=colors)
plt.xlabel('Correlation with Churn')
plt.title('Feature Correlation with Churn')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print('\nCorrelation with Churn:')
print(churn_corr.to_frame('Correlation').round(3))

## 6. Multi-feature Analysis

In [ ]:
# Scatter plot: Tenure vs Monthly Charges by Churn
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df['tenure'], df['MonthlyCharges'], 
                      c=df['Churn'].map({'No': 0, 'Yes': 1}),
                      cmap='RdYlGn_r', alpha=0.5, s=50)
plt.colorbar(scatter, label='Churn (0=No, 1=Yes)')
plt.xlabel('Tenure (months)')
plt.ylabel('Monthly Charges ($)')
plt.title('Tenure vs Monthly Charges colored by Churn Status')
plt.tight_layout()
plt.show()

In [ ]:
# Contract + Internet Service vs Churn
pivot_table = df.pivot_table(values='customerID', index='Contract', 
                             columns='InternetService', aggfunc='count', fill_value=0)

pivot_churn = df.groupby(['Contract', 'InternetService'])['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).unstack()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pivot_table, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Customer Count: Contract vs Internet Service')

sns.heatmap(pivot_churn, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=axes[1])
axes[1].set_title('Churn Rate (%): Contract vs Internet Service')

plt.tight_layout()
plt.show()

## 7. Key Insights & Conclusions

In [ ]:
# Summary statistics
print('='*60)
print('KEY INSIGHTS FROM EDA')
print('='*60)

# Overall churn rate
overall_churn = (df['Churn'] == 'Yes').mean() * 100
print(f'\n1. OVERALL CHURN RATE: {overall_churn:.1f}%')

# High churn segments
print('\n2. HIGH CHURN SEGMENTS:')
print(f'   - Month-to-month contracts: {df[df["Contract"]=="Month-to-month"]["Churn"].apply(lambda x: x=="Yes").mean()*100:.1f}%')
print(f'   - Fiber optic customers: {df[df["InternetService"]=="Fiber optic"]["Churn"].apply(lambda x: x=="Yes").mean()*100:.1f}%')
print(f'   - Electronic check payments: {df[df["PaymentMethod"]=="Electronic check"]["Churn"].apply(lambda x: x=="Yes").mean()*100:.1f}%')

# Low churn segments
print('\n3. LOW CHURN SEGMENTS:')
print(f'   - Two-year contracts: {df[df["Contract"]=="Two year"]["Churn"].apply(lambda x: x=="Yes").mean()*100:.1f}%')
print(f'   - No internet service: {df[df["InternetService"]=="No"]["Churn"].apply(lambda x: x=="Yes").mean()*100:.1f}%')

# Tenure impact
print('\n4. TENURE IMPACT:')
short_tenure_churn = df[df['tenure'] <= 12]['Churn'].apply(lambda x: x == 'Yes').mean() * 100
long_tenure_churn = df[df['tenure'] > 48]['Churn'].apply(lambda x: x == 'Yes').mean() * 100
print(f'   - New customers (0-12 months): {short_tenure_churn:.1f}% churn')
print(f'   - Loyal customers (48+ months): {long_tenure_churn:.1f}% churn')

# Charges impact
print('\n5. CHARGES IMPACT:')
high_charges_churn = df[df['MonthlyCharges'] > df['MonthlyCharges'].median()]['Churn'].apply(lambda x: x == 'Yes').mean() * 100
low_charges_churn = df[df['MonthlyCharges'] <= df['MonthlyCharges'].median()]['Churn'].apply(lambda x: x == 'Yes').mean() * 100
print(f'   - High monthly charges (>${df["MonthlyCharges"].median():.0f}): {high_charges_churn:.1f}% churn')
print(f'   - Low monthly charges (<=${df["MonthlyCharges"].median():.0f}): {low_charges_churn:.1f}% churn')

print('\n' + '='*60)

In [ ]:
# Recommendations based on EDA
print('RECOMMENDATIONS FOR MODEL DEVELOPMENT:')
print('-'*50)
print('''
1. IMPORTANT FEATURES TO INCLUDE:
   - Contract type (strongest predictor)
   - Tenure (negative correlation with churn)
   - Internet service type
   - Payment method
   - Monthly charges
   - Support services (TechSupport, OnlineSecurity)

2. CLASS IMBALANCE:
   - Dataset has ~26% churn rate
   - Consider SMOTE, class weights, or stratified sampling

3. FEATURE ENGINEERING IDEAS:
   - Create tenure groups (bins)
   - Total services count
   - Average monthly charge ratio
   - Automatic payment flag
   - Contract duration remaining

4. DATA CLEANING NEEDED:
   - Fix TotalCharges whitespace issues (11 rows)
   - Convert SeniorCitizen to categorical
''')